# 02 — Test bus package (Python wrapper)

Smoke test du package `src/bus/`. Là où `01_test_pubsub.ipynb` valide le **container Redis** (l'infra), ce notebook valide le **code Python** qui l'enrobe : factory de clients, fail-fast au boot, helpers publishers (5), TTL constants, throttle 200ms par symbole, et simulation des cycles d'écriture des 3 engines.

**Couvre** :
1. Imports du package `bus` (channels, get_redis, get_sync_redis, keys, publisher helpers)
2. Fail-fast quand `REDIS_URL` est absent de l'env
3. Les 5 publishers (`set_heartbeat`, `publish_tick`, `publish_account`, `publish_vol_update`, `publish_risk_update`)
4. Inspection des keys écrites + TTL conformes à `bus.keys`
5. Throttle `publish_tick` (1 PUBLISH max / 200ms / symbole, SET jamais throttlé)
6. Simulation des 3 engines (MarketDataEngine, VolEngine, RiskEngine) end-of-cycle

**Préreq** :
- Container `fxvol-redis` running (`.\scripts\start_stack.ps1`)
- `pip install redis` (déjà dans `requirements.txt`)

**Référence** : `src/bus/__init__.py`, `src/bus/keys.py`, `src/bus/channels.py`, `src/bus/publisher.py`, `src/bus/redis_client.py`.

## Setup — connexion + helpers

In [25]:
import os
import subprocess
import sys
import time
from pathlib import Path

# Walk up jusqu'à trouver src/ -- robuste quel que soit le CWD du kernel.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError(f"Cannot locate project root from {Path.cwd()}")
os.chdir(PROJECT_ROOT)
SRC = str(PROJECT_ROOT / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

os.environ["REDIS_URL"] = "redis://localhost:6380/0"

# docker start = idempotent, no compose interpolation
subprocess.run(["docker", "start", "fxvol-redis"], capture_output=True, text=True)

results = []
MARKER = f"BUS_TEST_{int(time.time())}"

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

print("CWD       :", PROJECT_ROOT)
print("REDIS_URL :", os.environ["REDIS_URL"])
print("Marker    :", MARKER)

CWD       : C:\Users\Valérian Darmenté\Documents\Python Project\fx-volatility-trading-system
REDIS_URL : redis://localhost:6380/0
Marker    : BUS_TEST_1777305072


## 1. Imports du package `bus`

**Ce que tu dois voir** : tous les symboles publics de `src/bus/` s'importent — clients (sync + async), keys templates, channels constants, helpers publishers. Les templates de keys se formatent correctement avec `.format(symbol=...)`.

In [26]:
print("== imports bus package ==")

try:
    from bus import channels, get_redis, get_sync_redis, keys
    from bus.publisher import (
        publish_account, publish_risk_update, publish_tick,
        publish_vol_update, reset_throttle_for_tests, set_heartbeat,
    )
    record("import bus.{channels,keys,get_redis,get_sync_redis}", True)
    record("import bus.publisher.{5 helpers + reset_throttle_for_tests}", True)
except ImportError as e:
    record("import bus.* helpers", False, str(e))
    raise

spot_key = keys.LATEST_SPOT.format(symbol="EURUSD")
record("keys.LATEST_SPOT format(symbol=...)", spot_key == "latest_spot:EURUSD", spot_key)
record("keys.TTL_SPOT = 30s", keys.TTL_SPOT == 30, f"got {keys.TTL_SPOT}")
record("channels.CH_TICKS = 'ticks'", channels.CH_TICKS == "ticks", channels.CH_TICKS)

== imports bus package ==
  [OK  ] import bus.{channels,keys,get_redis,get_sync_redis}
  [OK  ] import bus.publisher.{5 helpers + reset_throttle_for_tests}
  [OK  ] keys.LATEST_SPOT format(symbol=...)  | latest_spot:EURUSD
  [OK  ] keys.TTL_SPOT = 30s  | got 30
  [OK  ] channels.CH_TICKS = 'ticks'  | ticks


## 2. Fail-fast sans `REDIS_URL`

**Ce que tu dois voir** : `get_redis()` refuse de construire un client si `REDIS_URL` est absent de l'env. Crash au boot >>> service silencieusement KO pendant des heures en prod. C'est un design intentionnel du wrapper.

In [27]:
print("== fail-fast sans REDIS_URL ==")

from bus.redis_client import reset_clients_for_tests

saved = os.environ.pop("REDIS_URL")
reset_clients_for_tests()
fired = False
try:
    get_redis()
except RuntimeError as e:
    fired = True
    record("get_redis() raise RuntimeError quand REDIS_URL absent", True, str(e)[:60])
finally:
    os.environ["REDIS_URL"] = saved
    reset_clients_for_tests()

if not fired:
    record("get_redis() raise RuntimeError quand REDIS_URL absent", False, "no exception")

== fail-fast sans REDIS_URL ==
  [OK  ] get_redis() raise RuntimeError quand REDIS_URL absent  | REDIS_URL is not set. See .env.example for the expected form


## 3. Publisher helpers (5)

**Ce que tu dois voir** : les 5 helpers (`set_heartbeat`, `publish_tick`, `publish_account`, `publish_vol_update`, `publish_risk_update`) écrivent dans Redis sans exception. Les keys arrivent bien (vérification cellule suivante).

Note Jupyter : on utilise `await` direct (top-level async, IPython 7.x+) — pas `asyncio.run()` qui crash dans Jupyter.

In [28]:
print("== publisher helpers ==")

r = get_redis()
reset_throttle_for_tests()

try:
    await set_heartbeat(r, keys.ENGINE_MARKET_DATA)
    await set_heartbeat(r, keys.ENGINE_VOL)
    await set_heartbeat(r, keys.ENGINE_RISK)
    record("set_heartbeat × 3 engines", True)

    await publish_tick(r, "EURUSD", bid=1.0856, ask=1.0858, mid=1.0857)
    record("publish_tick (3 SET + 1 PUBLISH)", True)

    await publish_account(r, {"net_liq_usd": 125_000, "open_positions_count": 4})
    record("publish_account (1 SET + 1 PUBLISH)", True)

    await publish_vol_update(r, symbol="EURUSD",
        surface_data={"1M": {"atm": 7.5}, "3M": {"atm": 8.0}},
        signals_data=[{"tenor": "1M", "type": "CHEAP"}])
    record("publish_vol_update (2 SET + 1 PUBLISH)", True)

    await publish_risk_update(r,
        greeks={"delta_net": 1200, "vega_net": 500},
        pnl_curve={"spots": [1.08, 1.09], "pnls": [0, 100]})
    record("publish_risk_update (2 SET + 1 PUBLISH)", True)
except Exception as e:
    record("publisher helpers", False, str(e)[:80])

== publisher helpers ==
  [OK  ] set_heartbeat × 3 engines
  [OK  ] publish_tick (3 SET + 1 PUBLISH)
  [OK  ] publish_account (1 SET + 1 PUBLISH)
  [OK  ] publish_vol_update (2 SET + 1 PUBLISH)
  [OK  ] publish_risk_update (2 SET + 1 PUBLISH)


## 4. Keys + TTL conformes à `bus.keys`

**Ce que tu dois voir** : chaque key écrite par les publishers de § 3 a le bon TTL (déclaré dans `bus.keys`). Tolérance ±3s pour absorber le délai entre le SET et le check TTL.

In [29]:
print("== keys + TTL ==")

expected_ttls = [
    ("latest_spot:EURUSD", keys.TTL_SPOT),
    ("latest_bid:EURUSD", keys.TTL_BID_ASK),
    ("latest_ask:EURUSD", keys.TTL_BID_ASK),
    ("account_snapshot", keys.TTL_ACCOUNT),
    ("latest_vol_surface:EURUSD", keys.TTL_VOL_SURFACE),
    ("latest_signals:EURUSD", keys.TTL_SIGNALS),
    ("latest_greeks:portfolio", keys.TTL_GREEKS),
    ("latest_pnl_curve", keys.TTL_PNL_CURVE),
    ("heartbeat:market_data", keys.TTL_HEARTBEAT),
]

for key, expected in expected_ttls:
    out = subprocess.run(
        ["docker", "exec", "fxvol-redis", "redis-cli", "TTL", key],
        capture_output=True, text=True,
    )
    try:
        ttl = int(out.stdout.strip())
    except ValueError:
        ttl = -99
    # -2 = key absente / -1 = pas de TTL / sinon entier secondes restantes
    ok = expected - 3 <= ttl <= expected
    record(f"TTL {key}", ok, f"got {ttl}, expected ~{expected}")

== keys + TTL ==
  [OK  ] TTL latest_spot:EURUSD  | got 30, expected ~30
  [OK  ] TTL latest_bid:EURUSD  | got 30, expected ~30
  [OK  ] TTL latest_ask:EURUSD  | got 29, expected ~30
  [OK  ] TTL account_snapshot  | got 59, expected ~60
  [OK  ] TTL latest_vol_surface:EURUSD  | got 599, expected ~600
  [OK  ] TTL latest_signals:EURUSD  | got 599, expected ~600
  [OK  ] TTL latest_greeks:portfolio  | got 29, expected ~30
  [OK  ] TTL latest_pnl_curve  | got 29, expected ~30
  [OK  ] TTL heartbeat:market_data  | got 29, expected ~30


## 5. Tick throttle 200ms par symbole

**Ce que tu dois voir** : un burst de 50 ticks sur EURUSD émet **1 seul PUBLISH** (le 1er, le throttle bloque les 49 suivants car ils tombent dans la même fenêtre 200ms). Les 50 SET passent **toujours** — le cache `latest_spot` doit refléter le tick le plus récent.

**Pourquoi** : `publish_tick` enverrait sinon ~10000 messages/seconde au frontend en burst de marché ouvert. Le throttle limite à 5 msg/s/symbole pour ne pas saturer le WS.

In [30]:
print("== tick throttle ==")

reset_throttle_for_tests()
published = 0
swallowed = 0
for i in range(50):
    ok_pub = await publish_tick(r, "EURUSD", 1.0 + 1e-6 * i, 1.0 + 1e-6 * i, 1.0)
    if ok_pub:
        published += 1
    else:
        swallowed += 1

record("50 ticks burst -> 1 ou 2 PUBLISHs (throttle actif)", 1 <= published <= 2,
       f"{published} pub / {swallowed} throttled")

out = subprocess.run(["docker", "exec", "fxvol-redis", "redis-cli", "GET", "latest_spot:EURUSD"],
                     capture_output=True, text=True)
spot = out.stdout.strip()
record("cache latest_spot reflète bien le 50e tick", spot == "1.0", f"got {spot}")

== tick throttle ==
  [OK  ] 50 ticks burst -> 1 ou 2 PUBLISHs (throttle actif)  | 1 pub / 49 throttled
  [OK  ] cache latest_spot reflète bien le 50e tick  | got 1.0


## 6. Simulation des 3 engines end-of-cycle

**Ce que tu dois voir** : on simule ce que `MarketDataEngine`, `VolEngine` et `RiskEngine` font au end-of-cycle :
- MarketData : tick + account + heartbeat (toutes les 100ms en prod)
- VolEngine : surface + signals + heartbeat (toutes les ~180s)
- RiskEngine : greeks + pnl_curve + heartbeat (toutes les ~60s)

Test bonus de résilience : un client Redis cassé doit raise `ConnectionError` côté publisher (les engines en prod la swallow via `_REDIS_SWALLOW`, mais ce test prouve que le publisher LET la remonter — c'est l'engine qui décide si elle est fatale ou non).

In [31]:
print("== engines simulation end-of-cycle ==")

reset_throttle_for_tests()

try:
    await publish_tick(r, "EURUSD", 1.0856, 1.0858, 1.0857)
    await publish_account(r, {"net_liq_usd": 125_000, "open_positions_count": 4})
    await set_heartbeat(r, keys.ENGINE_MARKET_DATA)
    record("MarketDataEngine end-of-cycle", True)
except Exception as e:
    record("MarketDataEngine end-of-cycle", False, str(e)[:80])

try:
    await publish_vol_update(r, symbol="EURUSD",
        surface_data={"1M": {"atm": 7.5, "signal": "CHEAP"}, "3M": {"atm": 8.0, "signal": "FAIR"}},
        signals_data=[{"tenor": "1M", "signal": "CHEAP"}])
    await set_heartbeat(r, keys.ENGINE_VOL)
    record("VolEngine end-of-scan", True)
except Exception as e:
    record("VolEngine end-of-scan", False, str(e)[:80])

try:
    await publish_risk_update(r,
        greeks={"delta_net": 1200, "vega_net": 500, "pnl_total": 250},
        pnl_curve={"spots": [1.08, 1.09], "pnls": [0, 100]})
    await set_heartbeat(r, keys.ENGINE_RISK)
    record("RiskEngine end-of-cycle", True)
except Exception as e:
    record("RiskEngine end-of-cycle", False, str(e)[:80])

from unittest.mock import AsyncMock
from redis import exceptions as _redis_exc

broken = AsyncMock()
broken.set = AsyncMock(side_effect=_redis_exc.ConnectionError("Connection reset"))
broken.publish = AsyncMock()
raised = False
try:
    await publish_tick(broken, "EURUSD", 1.0, 1.0, 1.0)
except _redis_exc.ConnectionError:
    raised = True
record("publisher LAISSE remonter ConnectionError (engine catch)", raised,
       "bien : permet à l'engine de décider via _REDIS_SWALLOW")

== engines simulation end-of-cycle ==
  [OK  ] MarketDataEngine end-of-cycle
  [OK  ] VolEngine end-of-scan
  [OK  ] RiskEngine end-of-cycle
  [OK  ] publisher LAISSE remonter ConnectionError (engine catch)  | bien : permet à l'engine de décider via _REDIS_SWALLOW


## Récap final

Tableau OK/FAIL + cleanup final.

In [32]:
try:
    await r.aclose()
except Exception:
    pass

n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<55} STATUS  DETAIL")
print("-" * 100)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<55} {sym:<6}  {detail}")
print("-" * 100)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail:
    print("\n⚠ FAILs detected. Common causes:")
    print("  - container redis pas démarré        -> docker compose ps")
    print("  - REDIS_URL pointe ailleurs           -> doit être redis://localhost:6380/0")
    print("  - import bus.* échoue                 -> sys.path = src/, vérifier walk-up")
    print("  - TTL différent de bus.keys           -> drift entre publisher et keys.py")
    print("  - throttle 0 ou >2 publish            -> reset_throttle_for_tests() pas appelé")
else:
    print("\n✅ src/bus/ wrapper validé : imports + fail-fast + 5 publishers + TTL + throttle + 3 engines simulation.")


LABEL                                                   STATUS  DETAIL
----------------------------------------------------------------------------------------------------
import bus.{channels,keys,get_redis,get_sync_redis}     OK      
import bus.publisher.{5 helpers + reset_throttle_for_tests} OK      
keys.LATEST_SPOT format(symbol=...)                     OK      latest_spot:EURUSD
keys.TTL_SPOT = 30s                                     OK      got 30
channels.CH_TICKS = 'ticks'                             OK      ticks
get_redis() raise RuntimeError quand REDIS_URL absent   OK      REDIS_URL is not set. See .env.example for the expected form
set_heartbeat × 3 engines                               OK      
publish_tick (3 SET + 1 PUBLISH)                        OK      
publish_account (1 SET + 1 PUBLISH)                     OK      
publish_vol_update (2 SET + 1 PUBLISH)                  OK      
publish_risk_update (2 SET + 1 PUBLISH)                 OK      
TTL latest_spot:EUR